# Notebook 22g — Wu 2003 S-B3: pool the multi-machine widened-lognormal training banks

## Purpose

Pools every per-machine training chunk produced by `notebooks/22e`
(`data/wu2003_sbi_train_sb3_widelognormal_seed*_n*.npz`) into one training bank for
`notebooks/24d`. Unlike `notebooks/22d`'s pooling of two genuinely different priors (which needed
SNPE's atomic proposal-correction machinery, tagging each row's true generating distribution),
every chunk here is drawn from the exact same prior (same fixed `sigma`) — so a plain concatenation
is a mathematically valid i.i.d. sample from one distribution, no `proposal_kind` tagging needed.

**Not executed** — depends on `notebooks/22e` having been run (with distinct seeds) on however
many machines were actually used, and their output `.npz` files copied into one shared `data/`
folder first.

## Output file

- `data/wu2003_sbi_train_sb3_widelognormal_pooled_n{TOTAL}.npz`


## Section 1 — Registry: find every per-machine bank


In [ ]:
import pathlib, sys
import numpy as np

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT / 'src'))

from cstr_sbi.recycle.summaries import N_SUMMARIES_SB

DATA = ROOT / 'data'
EXPECTED_PRIOR_TAG = 'widelognormal_v1'
PATTERN = 'wu2003_sbi_train_sb3_widelognormal_seed*_n*.npz'

candidate_paths = sorted(p for p in DATA.glob(PATTERN) if not p.name.endswith('.partial.npz'))
print(f'Found {len(candidate_paths)} candidate bank(s) matching {PATTERN!r}:')
for p in candidate_paths:
    print(' ', p.name)


## Section 2 — Load and verify each source bank

Every source must declare the same `prior_tag`/`t_final_h`/`n_save`, and no two sources may share a `seed` (a machine accidentally re-run with the same `SB3_GEN_SEED` and its output copied twice).


In [ ]:
def load_and_verify(path):
    data = np.load(path)
    prior_tag = str(data['prior_tag']) if 'prior_tag' in data.files else None
    seed = int(data['seed']) if 'seed' in data.files else None
    t_final_h = float(data['t_final_h']) if 't_final_h' in data.files else None
    n_save = int(data['n_save']) if 'n_save' in data.files else None
    if prior_tag != EXPECTED_PRIOR_TAG:
        raise ValueError(
            f'{path.name}: prior_tag={prior_tag!r}, expected {EXPECTED_PRIOR_TAG!r}. Refusing to '
            'silently pool a bank drawn from a different prior.'
        )
    theta = np.asarray(data['thetas4'], dtype=np.float32)
    summary = np.asarray(data['summaries'], dtype=np.float32)
    assert summary.shape[1] == N_SUMMARIES_SB, f'{path.name}: unexpected summary dim {summary.shape[1]}'
    return {
        'path': path, 'seed': seed, 'theta': theta, 'summary': summary,
        't_final_h': t_final_h, 'n_save': n_save, 'rows': theta.shape[0],
    }


entries = [load_and_verify(p) for p in candidate_paths]

seeds = [e['seed'] for e in entries]
assert len(seeds) == len(set(seeds)), (
    f'Duplicate seeds found across banks: {seeds} -- a machine was likely re-run with the same '
    'SB3_GEN_SEED and its output copied in twice.'
)

t_finals = {e['t_final_h'] for e in entries}
n_saves = {e['n_save'] for e in entries}
assert len(t_finals) <= 1, f'Mismatched t_final_h across banks: {t_finals}'
assert len(n_saves) <= 1, f'Mismatched n_save across banks: {n_saves}'

for e in entries:
    print(f"seed={e['seed']:>4}  rows={e['rows']:>6}  {e['path'].name}")
print(f'TOTAL rows: {sum(e["rows"] for e in entries)}')


## Section 3 — Concatenate and save with provenance


In [ ]:
theta_all = np.concatenate([e['theta'] for e in entries], axis=0)
summary_all = np.concatenate([e['summary'] for e in entries], axis=0)
source_seeds = np.asarray([e['seed'] for e in entries], dtype=np.int64)
source_rows = np.asarray([e['rows'] for e in entries], dtype=np.int64)

OUT_PATH = DATA / f'wu2003_sbi_train_sb3_widelognormal_pooled_n{theta_all.shape[0]}.npz'
np.savez(
    OUT_PATH,
    thetas4=theta_all,
    summaries=summary_all,
    source_seeds=source_seeds,
    source_rows=source_rows,
    structure='S-B3',
    prior_tag=EXPECTED_PRIOR_TAG,
    t_final_h=next(iter(t_finals)),
    n_save=next(iter(n_saves)),
)
print(f'saved {OUT_PATH.name}: {theta_all.shape[0]} rows from {len(entries)} source bank(s)')


## Section 4 — Round-trip verification and acceptance criteria


In [ ]:
rt = np.load(OUT_PATH)
criteria = [
    ('pooled row count = sum of sources',
     int(rt['thetas4'].shape[0]) == int(source_rows.sum()),
     f"{rt['thetas4'].shape[0]} vs {source_rows.sum()}"),
    ('all summaries finite', bool(np.isfinite(rt['summaries']).all()), 'finite check'),
    ('summary dimension is S-B', int(rt['summaries'].shape[1]) == N_SUMMARIES_SB,
     str(rt['summaries'].shape[1])),
    ('no duplicate source seeds',
     len(set(rt['source_seeds'].tolist())) == len(rt['source_seeds']),
     rt['source_seeds'].tolist()),
]
for name, passed, value in criteria:
    print(f"{'PASS' if passed else 'FAIL'} | {name}: {value}")

print()
print("Next step: notebooks/24d trains SNPE on this pooled bank with a single fixed prior")
print("(mixed_wide_prior_4d_sb3(sigma=LOGN_SIGMA_SB3_4D_WIDE)) and evaluates against")
print("notebooks/22f's held-out N=800 confirm set.")
